# 事業案の検証ツール

インストールも設定も要りません。**セルの左にある丸い再生ボタン（▶）を押すだけ**です。

上から順に、2つの検証を実行します。

| | 何を確かめるか | 準備 | 時間 |
|---|---|---|---|
| **1** | 補助金案（②）に、まだ隙間があるか | 不要 | 約3分 |
| **2** | 広告宣伝費DB（①）を作るべきか | キー取得（5分） | 約20分 |

**まず1から実行してください。** 準備が要らないので、動く感覚がすぐ掴めます。


---
## 1. 補助金案の検証

何も準備は要りません。下のセルの**▶を押すだけ**です。

**見るポイントは1つ。** 出力の最後に出る「**「全国」扱い: ◯件（◯%）**」という数字です。

- この比率が**高い**（6割超）→ 国の施策ばかりで、市区町村の補助金は手つかず＝**案②は作り直せる**
- この比率が**低い**→ 地域の補助金も十分載っている＝**案②は捨てる**


In [ ]:
#!/usr/bin/env python3
"""案②を生き返らせられるかを判定する1本：jGrants の中身を実測する。

背景
----
「全国の補助金を検索できる」はもう商品にならない（jGrants APIが認証不要で公開され、
既存の検索サイトも個人開発のSaaSも既にある）。生き返る余地があるとすれば次の2点だけ。

    仮説A: jGrants は国の施策が中心で、市区町村の独自補助金はほとんど載っていない
           → 載っていない領域＝手つかず。そこだけを取りに行けば商品になる
    仮説B: レコードに更新日時があり、差分検知（新着・条件変更の通知）が実装できる
           → 「検索」ではなく「プッシュ」に商品を寄せられる

このスクリプトは、その2つを実データで確かめる。
A が偽（自治体の補助金も網羅されている）なら、案②は完全に終わり。

使い方
------
    python3 scripts/jgrants_probe.py

    認証不要・追加パッケージ不要。数分で終わる。
"""

from __future__ import annotations

import json
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import Counter

BASE = "https://api.jgrants-portal.go.jp/exp"

# 全件一括取得ができない仕様なので、広めのキーワードで引いて id で重複排除する。
KEYWORDS = [
    "事業", "補助", "支援", "促進", "整備", "導入", "開発", "改善",
    "設備", "雇用", "人材", "研究", "環境", "観光", "農業", "医療",
    "創業", "販路", "デジタル", "省エネ", "子育て", "移住",
]

INTERVAL = 1.0


def get(path: str, params: dict | None = None) -> dict | None:
    url = f"{BASE}{path}"
    if params:
        url += "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "jgrants-probe/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as res:
            return json.loads(res.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        print(f"  ! HTTP {e.code} {url}", file=sys.stderr)
    except Exception as e:
        print(f"  ! {e} {url}", file=sys.stderr)
    return None


def main() -> None:
    print("■ キーワードを回して全件を集めます（重複はidで排除）\n")

    records: dict[str, dict] = {}
    for kw in KEYWORDS:
        payload = get(
            "/v1/public/subsidies",
            {"keyword": kw, "sort": "created_date", "order": "DESC", "acceptance": "0"},
        )
        time.sleep(INTERVAL)
        if not payload:
            continue
        # 仕様変更に備えて、結果配列のキー名を決め打ちしない
        items = payload.get("result") or payload.get("results") or []
        new = 0
        for it in items:
            key = it.get("id") or it.get("subsidy_id") or json.dumps(it, sort_keys=True)[:64]
            if key not in records:
                records[key] = it
                new += 1
        print(f"  {kw:<6} 取得 {len(items):>4} 件 / 新規 {new:>4} 件 / 累計 {len(records)}")

    if not records:
        sys.exit("\n1件も取得できませんでした。API仕様が変わった可能性があります。")

    sample = next(iter(records.values()))
    print("\n" + "=" * 60)
    print("■ レコードのフィールド一覧（仮説Bの判定材料）")
    print("=" * 60)
    for k, v in sample.items():
        shown = str(v)
        if len(shown) > 54:
            shown = shown[:54] + "…"
        print(f"  {k:<32} {shown}")

    date_fields = [k for k in sample if any(t in k.lower() for t in ("date", "time", "updated", "created"))]
    print("\n  日付・更新らしきフィールド:", ", ".join(date_fields) or "なし")
    if date_fields:
        print("  → 差分検知は実装できる。仮説B は成立。")
    else:
        print("  → 更新日時がない。全件を毎回保存して自前で差分を取る必要がある。")

    # 仮説A: 実施主体・対象地域の分布
    area_key = None
    for cand in ("target_area_search", "target_area_detail", "prefecture", "area"):
        if cand in sample:
            area_key = cand
            break

    print("\n" + "=" * 60)
    print("■ 対象地域の分布（仮説Aの判定材料）")
    print("=" * 60)
    if area_key:
        c = Counter()
        for r in records.values():
            v = r.get(area_key)
            c[str(v) if v else "(空)"] += 1
        national = sum(n for k, n in c.items() if "全国" in k)
        print(f"  対象地域フィールド: {area_key}")
        for k, n in c.most_common(20):
            print(f"    {k:<28} {n:>5} 件")
        print(f"\n  「全国」扱い: {national} 件 / 全体 {len(records)} 件"
              f" （{national/len(records)*100:.1f}%）")
        if national / len(records) > 0.6:
            print("  → 国の施策が中心。市区町村の独自補助金は手つかずの可能性が高い＝仮説A成立")
        else:
            print("  → 地域単位の補助金もかなり載っている。案②の隙間は狭い＝仮説A不成立の疑い")
    else:
        print("  対象地域を表すフィールドが見つからない。詳細APIで確認する必要がある。")

    # 実施機関が取れるか（自治体独自かどうかの直接の判定材料）
    print("\n" + "=" * 60)
    print("■ 詳細APIのフィールド（実施機関が取れるか）")
    print("=" * 60)
    sid = sample.get("id")
    if sid:
        detail = get(f"/v2/public/subsidies/id/{sid}")
        if detail:
            items = detail.get("result") or detail.get("results") or []
            d = items[0] if isinstance(items, list) and items else detail
            for k in list(d.keys())[:40]:
                print(f"  {k}")
        else:
            print("  詳細APIの取得に失敗")
    else:
        print("  id が取れないため詳細APIを呼べない")

    print("\n" + "=" * 60)
    print(f"総ユニーク件数: {len(records)}")
    print("=" * 60)
    print("""
判定の読み方
  ・総件数が既存サービスの掲載数（1万件規模を謳うものがある）を大きく下回り、
    かつ「全国」比率が高い → jGrants に載らない自治体独自の領域が残っている＝案②は作り直せる
  ・地域単位の補助金も十分載っている → 差別化の余地がない＝案②は捨てる
""")


main()


---
## 2. 広告宣伝費DBの検証

### 先にキーを取ります（無料・5分）

1. <https://api.edinet-fsa.go.jp/api/auth/index.aspx?mode=1> を開く
2. 所属・氏名・電話番号・メールアドレスを入力して登録
3. 届いたメールのリンクをクリックして認証
4. ログインして、マイページから**APIキーを発行**
5. 表示された長い英数字をコピー

> ポップアップがブロックされる場合があります。ブロックされたら、ブラウザのアドレスバー右端に出る通知から許可してください。

### そのあと

下のセルの **EDINET_KEY** の欄にキーを貼り付けて、▶を押します。

**まず SAMPLE を 30 にして試すことを勧めます**（約4分）。ちゃんと動くと分かってから150に上げてください。

**見るポイントは最後の「判定」の1行だけ**です。作る／捨てるまで出ます。明細のCSVは自動でダウンロードされます。


In [ ]:
#@title ← この左の丸い再生ボタンを押すと実行されます { display-mode: "form" }
#@markdown EDINETで取得したキーを、下の欄に貼り付けてください。
EDINET_KEY = ""  #@param {type:"string"}
#@markdown 調べる件数（150で約20分。まず30で試すのも可）
SAMPLE = 150  #@param {type:"integer"}
#@markdown 遡る日数（有報は6月に集中するので400推奨）
DAYS = 400  #@param {type:"integer"}

#!/usr/bin/env python3
"""案①の生死を決める1本：有価証券報告書のうち、広告宣伝費を開示している企業の割合を数える。

背景
----
広告宣伝費は EDINET 標準タクソノミの要素 `jppfs_cor:AdvertisingExpensesSGA` として
定義されているので、開示されてさえいれば機械的に取れる（表記ゆれの名寄せが不要）。
残る唯一の未知数が「そもそも何割の企業が販管費の内訳を開示しているか」。
IFRS・米国基準の提出会社や、内訳を注記に出さない会社があるため、実測しないと分からない。

判断の目安（本文の基準）
    網羅率 60% 以上 → 商品として成立する。作る
    30〜60%        → 業種を絞れば成立しうる。業種別の内訳を見て判断
    30% 以下       → 捨てる

使い方
------
    1. https://api.edinet-fsa.go.jp/ で Subscription-Key を取得（無料・即時）
    2. export EDINET_KEY=取得したキー
    3. python3 scripts/edinet_coverage.py --days 400 --sample 200

    追加パッケージ不要（標準ライブラリのみ）。--sample 200 でおよそ15〜25分。

注意
----
レート制限は公式に明示されていないため、既定で 1.2 秒間隔に抑えている。
429 や 403 が返り始めたら --interval を上げること。
"""

from __future__ import annotations

import argparse
import csv
import datetime as dt
import io
import json
import os
import random
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from collections import Counter

API = "https://api.edinet-fsa.go.jp/api/v2"

# 有価証券報告書。訂正有報(130)は本体と重複するので除く。
DOC_TYPE_YUHO = "120"

# 数えたい要素。CSV内の「要素ID」列とこの文字列を突き合わせる。
TARGET = "jppfs_cor:AdvertisingExpensesSGA"

# 参考として同時に数える、販管費の内訳が開示されているかの傍証。
REFERENCE = [
    "jppfs_cor:SalariesAndAllowancesSGA",        # 給料及び手当
    "jppfs_cor:ProvisionForBonusesSGA",          # 賞与引当金繰入額
    "jppfs_cor:DepreciationSGA",                 # 減価償却費
    "jppfs_cor:SellingExpensesSGA",              # 販売費
]


def fetch(url: str, key: str, timeout: int = 60) -> bytes:
    sep = "&" if "?" in url else "?"
    full = f"{url}{sep}Subscription-Key={urllib.parse.quote(key)}"
    req = urllib.request.Request(full, headers={"User-Agent": "coverage-probe/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as res:
        return res.read()


def list_filings(day: dt.date, key: str) -> list[dict]:
    """その日に提出された有報のうち、CSV(XBRL)が取得できるものを返す。"""
    url = f"{API}/documents.json?date={day.isoformat()}&type=2"
    try:
        payload = json.loads(fetch(url, key).decode("utf-8"))
    except urllib.error.HTTPError as e:
        if e.code in (401, 403):
            sys.exit(f"認証エラー({e.code})。EDINET_KEY を確認してください。")
        print(f"  ! {day} 一覧取得に失敗 HTTP {e.code}", file=sys.stderr)
        return []
    except Exception as e:  # ネットワーク断など
        print(f"  ! {day} 一覧取得に失敗 {e}", file=sys.stderr)
        return []

    out = []
    for r in payload.get("results") or []:
        if r.get("docTypeCode") != DOC_TYPE_YUHO:
            continue
        # csvFlag が "1" のものだけが type=5 で取得できる
        if str(r.get("csvFlag")) != "1":
            continue
        out.append(
            {
                "docID": r.get("docID"),
                "edinetCode": r.get("edinetCode"),
                "filerName": r.get("filerName"),
                "submitDate": day.isoformat(),
            }
        )
    return out


def elements_in_document(doc_id: str, key: str) -> set[str] | None:
    """XBRL の CSV を取得し、含まれている要素IDの集合を返す。失敗時 None。"""
    url = f"{API}/documents/{doc_id}?type=5"
    try:
        blob = fetch(url, key, timeout=180)
    except Exception as e:
        print(f"  ! {doc_id} 本体取得に失敗 {e}", file=sys.stderr)
        return None

    found: set[str] = set()
    try:
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            members = [n for n in z.namelist() if n.lower().endswith(".csv")]
            for name in members:
                raw = z.read(name)
                # EDINET の XBRL CSV は UTF-16 のタブ区切り
                try:
                    text = raw.decode("utf-16")
                except UnicodeError:
                    text = raw.decode("utf-8", errors="replace")
                reader = csv.reader(io.StringIO(text), delimiter="\t")
                for row in reader:
                    if row:
                        found.add(row[0].strip())
    except zipfile.BadZipFile:
        print(f"  ! {doc_id} ZIPとして読めない（PDFのみの可能性）", file=sys.stderr)
        return None
    return found


class Args:
    pass


def main() -> None:
    args = Args()
    args.days = DAYS
    args.sample = SAMPLE
    args.interval = 0.8
    args.out = "edinet_coverage.csv"
    args.seed = 42

    key = EDINET_KEY.strip()
    if not key:
        raise SystemExit("上の EDINET_KEY 欄にキーを貼り付けてから、もう一度実行してください。")

    today = dt.date.today()
    print(f"■ 直近 {args.days} 日の有価証券報告書を列挙します")

    filings: list[dict] = []
    for i in range(args.days):
        day = today - dt.timedelta(days=i)
        # 提出は平日に集中するので土日はスキップして時間を節約
        if day.weekday() >= 5:
            continue
        got = list_filings(day, key)
        if got:
            filings.extend(got)
            print(f"  {day} … {len(got)}件 (累計 {len(filings)})")
        time.sleep(args.interval)

    if not filings:
        sys.exit("有報が1件も取れませんでした。日付範囲かキーを確認してください。")

    # 同一企業が複数回出てくることがあるので EDINETコードで一意化
    uniq: dict[str, dict] = {}
    for f in filings:
        uniq.setdefault(f["edinetCode"] or f["docID"], f)
    pool = list(uniq.values())
    print(f"\n■ 有報 {len(filings)} 件 / 企業 {len(pool)} 社")

    random.seed(args.seed)
    sample = random.sample(pool, min(args.sample, len(pool)))
    print(f"■ うち {len(sample)} 社の中身を確認します\n")

    rows = []
    hit = 0
    checked = 0
    ref_counter: Counter[str] = Counter()

    for n, f in enumerate(sample, 1):
        els = elements_in_document(f["docID"], key)
        time.sleep(args.interval)
        if els is None:
            continue
        checked += 1
        has = TARGET in els
        hit += has
        for r in REFERENCE:
            if r in els:
                ref_counter[r] += 1
        rows.append(
            {
                "edinetCode": f["edinetCode"],
                "filerName": f["filerName"],
                "docID": f["docID"],
                "submitDate": f["submitDate"],
                "hasAdvertisingExpensesSGA": int(has),
            }
        )
        mark = "○" if has else "×"
        rate = hit / checked * 100
        print(f"  [{n:>4}/{len(sample)}] {mark} {(f['filerName'] or '')[:28]:<28} 途中経過 {rate:5.1f}%")

    with open(args.out, "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()) if rows else ["edinetCode"])
        w.writeheader()
        w.writerows(rows)

    print("\n" + "=" * 56)
    print(f"確認できた社数        : {checked}")
    print(f"広告宣伝費あり        : {hit}")
    coverage = hit / checked * 100 if checked else 0.0
    print(f"網羅率                : {coverage:.1f}%")
    print("-" * 56)
    print("参考（販管費内訳の開示状況）")
    for r in REFERENCE:
        c = ref_counter[r]
        print(f"  {r:<44} {c/checked*100 if checked else 0:5.1f}%")
    print("=" * 56)

    if coverage >= 60:
        verdict = "作る。網羅率は商品として十分。"
    elif coverage >= 30:
        verdict = "業種を絞れば成立しうる。明細CSVを業種別に見てから判断。"
    else:
        verdict = "捨てる。この網羅率ではデータ商品にならない。"
    print(f"\n判定: {verdict}")
    print(f"明細: {args.out}")


main()

# 明細CSVを手元にダウンロード
try:
    from google.colab import files
    files.download("edinet_coverage.csv")
except Exception:
    pass


---
## 終わったら

出力の最後のほう（「網羅率」と「判定」、そして「全国」の比率）を、**そのままコピーして貼ってください。**数字を読んで、①と②を作るか捨てるか確定させます。

### うまくいかないとき

- **赤い文字がたくさん出る** → 出力の最後の5行だけ貼ってください。こちらで直します。
- **認証エラー(401/403)** → キーの貼り付けミスか、キー発行がまだ完了していません。
- **途中で止まる** → Colabは放置すると切断されます。タブを開いたままにしてください。
